In [51]:
import librosa
import librosa.display
import soundfile as sf
import pretty_midi
import museval
from scipy.io import wavfile

from musicProcessing import MusicProcessingProvider as mp

from tqdm import tqdm
import os
import matplotlib.pyplot as plt
import numpy as np

from pandas import DataFrame
import pandas as pd

In [52]:
def get_source_activity(midi_path, sr, hop_length, max_frames):
    # loads individual source midi and returns a np.array with all the active time frame indexes
    pm = pretty_midi.PrettyMIDI(midi_path)
    active_frames = np.asarray([], dtype=int)
    for instrument in pm.instruments:
        for note in instrument.notes:
            start_frame = np.floor(note.start * sr / hop_length).astype(int)
            end_frame = np.floor(note.end * sr / hop_length).astype(int)
            active_frames = np.union1d(active_frames, np.arange(start_frame, end_frame))
    # clip to max_frames
    active_frames = active_frames[active_frames < max_frames]
    return active_frames

In [53]:
# 1. Modify the eval function to accept arrays of all stems
def eval_full_track(estimated_audios, reference_audios, win_length, hop_length):
    """
    estimated_audios: list of 1D arrays of the estimated stems
    reference_audios: list of 1D arrays of the reference stems
    """
    # Pad/truncate to ensure equal lengths across all accumulated stems
    min_len = min(min(len(a) for a in estimated_audios), min(len(r) for r in reference_audios))
    
    # Format to (n_sources, n_samples, n_channels)
    est_matrix = np.stack([a[:min_len, np.newaxis] for a in estimated_audios], axis=0)
    ref_matrix = np.stack([r[:min_len, np.newaxis] for r in reference_audios], axis=0)

    # Evaluate everything together so SIR can compute cross-interference
    sdr, isr, sir, sar = museval.evaluate(ref_matrix, est_matrix, win=win_length, hop=hop_length)
    
    # Returns shapes: (n_sources, n_frames, n_channels)
    # Take median across frames (axis 1)
    return np.nanmedian(sdr, axis=1), np.nanmedian(sir, axis=1), np.nanmedian(sar, axis=1)

In [54]:
baby_slakh_path = "/home/bordduk69/Nextcloud/NTNU/10. Vår2026/Master/Code/babyslakh_16k/"
results_path = "/home/bordduk69/Nextcloud/NTNU/10. Vår2026/Master/Code/REBEAT/experiments/results/exp3/"
repetiveness_scores_path = "/home/bordduk69/Nextcloud/NTNU/10. Vår2026/Master/Code/REBEAT/repetiveness.csv"

repetiveness_df = pd.read_csv(repetiveness_scores_path)
#drop all cols but track_id, stem_id and score
repetiveness_df = repetiveness_df[["track_id", "stem_id", "score"]]
# sort by track name then by source stem
repetiveness_df = repetiveness_df.sort_values(by=["track_id", "stem_id"])

nfft = 2048
win_length = 2048
hop_length = 512
window_type = "hann"

repetiveness_df.head()

,track_id,stem_id,score
118,Track00001,S00,0.782380
86,Track00001,S01,0.815883
159,Track00001,S02,0.712629
69,Track00001,S03,0.843499
129,Track00001,S04,0.761250


In [55]:
tracks = os.listdir(baby_slakh_path)
tracks.sort()

results = DataFrame(columns=["track_id", "stem_id", "sdr", "sir", "sar", "repetiveness_score"])

for track in tqdm(tracks[:2]):
    track_path = os.path.join(baby_slakh_path, track)
    
    # create folder for results if it doesn't exist
    track_results_path = os.path.join(results_path, track)
    os.makedirs(track_results_path, exist_ok=True)

    mixture_path = os.path.join(track_path, "mix.wav")
    stems_path = os.path.join(track_path, "stems")
    midi_path = os.path.join(track_path, "MIDI")
    
    music_processor = mp(path=mixture_path, duration=None, nfft=nfft, hop_length=hop_length, win_length=win_length, window_type=window_type)
    
    track_est_audios = []
    track_ref_audios = []
    track_stem_names = []
    
    # Get all clean wav files and sort them to guarantee a predictable sequence
    wav_files = sorted([f for f in os.listdir(stems_path) if f.endswith(".wav") and not f.startswith(".")])
    
    for wav_file in wav_files:
        stem_name = wav_file.replace(".wav", "")
        midi_file = f"{stem_name}.mid"
        
        stem_wave_path = os.path.join(stems_path, wav_file)
        stem_midi_path = os.path.join(midi_path, midi_file)
        
        # Guard against a missing MIDI file
        if not os.path.exists(stem_midi_path):
            print(f"Warning: Missing MIDI file for {stem_wave_path}. Skipping.")
            continue
            
        active_frames = get_source_activity(stem_midi_path, music_processor.sr, hop_length, music_processor.magnitude.shape[1])
        
        if len(active_frames) == 0:
            # Instead of crashing your entire pipeline, log it and move to the next stem
            print(f"Skipping {stem_name} in {track}: No active frames found in MIDI.")
            continue
        
        # perform median filtering on the active frames
        W = music_processor.median_filter(frames=active_frames)
        mask = music_processor.calc_mask(W)

        source_data = music_processor.apply_mask(music_processor.stft, mask)
        source_data = music_processor.calc_iSTFT(source_data, hop_length=hop_length, win_length=win_length, window_type=window_type)
        
        # save the estimated source
        estimated_path = os.path.join(track_results_path, f"{stem_name}_estimated.wav")
        sf.write(estimated_path, source_data, music_processor.sr)
        
        # Load and append to our evaluation batch
        _, est_audio = wavfile.read(estimated_path)
        _, ref_audio = wavfile.read(stem_wave_path)
        
        track_est_audios.append(est_audio)
        track_ref_audios.append(ref_audio)
        track_stem_names.append(stem_name)

    # --- Run evaluation once per track ---
    # Only run if we actually successfully processed stems for this track
    if track_est_audios:
        track_sdrs, track_sirs, track_sars = eval_full_track(track_est_audios, track_ref_audios, win_length, hop_length)

        # --- Store results into your dataframe ---
        for idx, stem_name in enumerate(track_stem_names):
            sdr_val = track_sdrs[idx]
            sir_val = track_sirs[idx]
            sar_val = track_sars[idx]
            
            # Get your repetiveness score...
            repetiveness_score = repetiveness_df[(repetiveness_df["track_id"] == track) & (repetiveness_df["stem_id"] == stem_name)]["score"].values
            repetiveness_score = repetiveness_score[0] if len(repetiveness_score) > 0 else None
            
            row = DataFrame({
                "track_id": [track],
                "stem_id": [stem_name],
                "sdr": [sdr_val],
                "sir": [sir_val],
                "sar": [sar_val],
                "repetiveness_score": [repetiveness_score]
            })
            results = pd.concat([results, row], ignore_index=True)
results.to_csv(os.path.join(results_path, "evaluation_results.csv"), index=False)

  0%|          | 0/2 [00:00<?, ?it/s]/home/bordduk69/Nextcloud/NTNU/10. Vår2026/Master/Code/REBEAT/.venv/lib/python3.14/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,
 50%|█████     | 1/2 [01:11<01:11, 71.76s/it]/home/bordduk69/Nextcloud/NTNU/10. Vår2026/Master/Code/REBEAT/.venv/lib/python3.14/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,
100%|██████████| 2/2 [02:11<00:00, 65.65s/it]
